In [78]:
import json
from pathlib import Path

import pandas as pd

In [79]:
LOCAL_PATH = Path("../data") / "bronze_data.json"

def create_dataframe(local_path: str) -> pd.DataFrame:
    path = local_path

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with open(path) as f:
        data = json.load(f)

    df = pd.json_normalize(data)
    return df

df = create_dataframe(LOCAL_PATH)


In [80]:
def normalize_data_items_columns(df: pd.DataFrame) -> pd.DataFrame:
    df_exploded = df.explode('data.items').reset_index(drop=True)

    df_data_items = pd.json_normalize(df_exploded['data.items'])
    
    df = pd.concat(
        [df_exploded.drop(columns="data.items"), df_data_items],
        axis=1,
    )

    return df

df = normalize_data_items_columns(df)

In [81]:
unnecessary_columns = [
    'requestId', 'executionId', 'data.source', 'data.type', 'data.url', 
    'data.requestUrl', 'data.query', 'data.totalResults', 'data.primaryResults', 
    'data.page', 'data.resultsPerPage', 'data.offset', 'data.nextPage', 
    'data.nextPageUrl', 'aggregateRating'
]

def drop_columns(df: pd.DataFrame, unnecessary_columns: list) -> pd.DataFrame:
    df = df.drop(columns=unnecessary_columns, errors="ignore")

    # df = df.dropna(how='all', axis=1)
    # df = df.loc[:, df.nunique() > 1]

    print(f"Colunas atuais ({len(df.columns)}): {df.columns.tolist()}")
    print(f"Toda nula: {df.columns[df.isna().all()].tolist()}")
    print(f"Constante: {[c for c in df.columns if df[c].nunique(dropna=True) == 1]}")

    return df

df = drop_columns(df, unnecessary_columns)

df.info(verbose=True, show_counts=True)

duplicated_counts = pd.DataFrame({
    'total_records_not_null': df.notna().sum(),
    'distinct_values': df.nunique()
})
print("\nDuplicated counts per column:")
print(duplicated_counts)

Colunas atuais (23): ['data.extractedAt', 'url', 'imageUrl', 'sku', 'categoryId', 'domainId', 'name', 'condition', 'currency', 'currencyRaw', 'price', 'highlight', 'isBestSeller', 'isPowerSeller', 'powerSellerStatusTitle', 'ean', 'sellerId', 'sellerName', 'sellerCity', 'sellerState', 'sellerCountry', 'aggregateRating.rating', 'aggregateRating.reviewCount']
Toda nula: ['powerSellerStatusTitle', 'ean', 'sellerId', 'sellerCity', 'sellerState', 'sellerCountry', 'aggregateRating.reviewCount']
Constante: ['categoryId', 'domainId', 'currency', 'currencyRaw', 'isBestSeller', 'isPowerSeller']
<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   data.extractedAt             184 non-null    str    
 1   url                          184 non-null    str    
 2   imageUrl                     184 non-null    str    
 3   sku                 

In [82]:
required = ["url", "sku"]

def handle_nulls_and_empty(df: pd.DataFrame) -> pd.DataFrame:
    if all(col in df.columns for col in required):
        df = df.query("url.notna() and sku.notna() and url != '' and sku != ''").reset_index(drop=True)

    return df

def handle_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    if all(col in df.columns for col in required):
        df = df.drop_duplicates(subset=['url', 'sku']).reset_index(drop=True)

    return df


df = handle_nulls_and_empty(df)
df = handle_duplicates(df)

df.info(verbose=True, show_counts=True)

duplicated_counts = pd.DataFrame({
    'total_records_not_null': df.notna().sum(),
    'distinct_values': df.nunique()
})
print("\nDuplicated counts per column:")
print(duplicated_counts)

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   data.extractedAt             181 non-null    str    
 1   url                          181 non-null    str    
 2   imageUrl                     181 non-null    str    
 3   sku                          181 non-null    str    
 4   categoryId                   178 non-null    str    
 5   domainId                     181 non-null    str    
 6   name                         181 non-null    str    
 7   condition                    20 non-null     str    
 8   currency                     181 non-null    str    
 9   currencyRaw                  181 non-null    str    
 10  price                        181 non-null    float64
 11  highlight                    30 non-null     str    
 12  isBestSeller                 181 non-null    bool   
 13  isPowerSeller                18

In [83]:
def impute_highlight_column(df: pd.DataFrame) -> pd.DataFrame:
    if 'highlight' in df.columns:
        df['highlight'] = df['highlight'].fillna('Sem destaque')

    return df

df = impute_highlight_column(df)

df.info(verbose=True, show_counts=True)

duplicated_counts = pd.DataFrame({
    'total_records_not_null': df.notna().sum(),
    'distinct_values': df.nunique()
})
print("\nDuplicated counts per column:")
print(duplicated_counts)

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   data.extractedAt             181 non-null    str    
 1   url                          181 non-null    str    
 2   imageUrl                     181 non-null    str    
 3   sku                          181 non-null    str    
 4   categoryId                   178 non-null    str    
 5   domainId                     181 non-null    str    
 6   name                         181 non-null    str    
 7   condition                    20 non-null     str    
 8   currency                     181 non-null    str    
 9   currencyRaw                  181 non-null    str    
 10  price                        181 non-null    float64
 11  highlight                    181 non-null    str    
 12  isBestSeller                 181 non-null    bool   
 13  isPowerSeller                18

In [84]:
def clean_invalid_values(df: pd.DataFrame) -> pd.DataFrame:
    if 'price' in df.columns:
        df = df.query("price > 0").reset_index(drop=True)

    if 'aggregateRating.rating' in df.columns:
        valid = (df['aggregateRating.rating'] >= 1) & (df['aggregateRating.rating'] <= 5)
        df['aggregateRating.rating'] = df['aggregateRating.rating'].where(valid, other=pd.NA)

    return df

df = clean_invalid_values(df)

df.info(verbose=True, show_counts=True)

duplicated_counts = pd.DataFrame({
    'total_records_not_null': df.notna().sum(),
    'distinct_values': df.nunique()
})
print("\nDuplicated counts per column:")
print(duplicated_counts)

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   data.extractedAt             181 non-null    str    
 1   url                          181 non-null    str    
 2   imageUrl                     181 non-null    str    
 3   sku                          181 non-null    str    
 4   categoryId                   178 non-null    str    
 5   domainId                     181 non-null    str    
 6   name                         181 non-null    str    
 7   condition                    20 non-null     str    
 8   currency                     181 non-null    str    
 9   currencyRaw                  181 non-null    str    
 10  price                        181 non-null    float64
 11  highlight                    181 non-null    str    
 12  isBestSeller                 181 non-null    bool   
 13  isPowerSeller                18

In [85]:
columns_names_to_rename = {
    "data.extractedAt": "extracted_at",
    "imageUrl": "image_url",
    "categoryId": "category_id",
    "domainId": "domain_id",
    "currencyRaw": "currency_raw",
    "isBestSeller": "is_best_seller",
    "isPowerSeller": "is_power_seller",
    "powerSellerStatusTitle": "power_seller_status_title",
    "sellerId": "seller_id",
    "sellerName": "seller_name",
    "sellerCity": "seller_city",
    "sellerState": "seller_state",
    "sellerCountry": "seller_country",
    "aggregateRating.rating": "rating",
    "aggregateRating.reviewCount": "review_count",
}

def rename_columns(df: pd.DataFrame, columns_names_to_rename: dict) -> pd.DataFrame:
    df = df.rename(columns=columns_names_to_rename, errors="ignore")
    
    return df

df = rename_columns(df, columns_names_to_rename)

df.info(verbose=True, show_counts=True)

duplicated_counts = pd.DataFrame({
    'total_records_not_null': df.notna().sum(),
    'distinct_values': df.nunique()
})
print("\nDuplicated counts per column:")
print(duplicated_counts)

<class 'pandas.DataFrame'>
RangeIndex: 181 entries, 0 to 180
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   extracted_at               181 non-null    str    
 1   url                        181 non-null    str    
 2   image_url                  181 non-null    str    
 3   sku                        181 non-null    str    
 4   category_id                178 non-null    str    
 5   domain_id                  181 non-null    str    
 6   name                       181 non-null    str    
 7   condition                  20 non-null     str    
 8   currency                   181 non-null    str    
 9   currency_raw               181 non-null    str    
 10  price                      181 non-null    float64
 11  highlight                  181 non-null    str    
 12  is_best_seller             181 non-null    bool   
 13  is_power_seller            181 non-null    bool   
 14  power